In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-ml

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT08 — Exercise 6: Transformer Architecture
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build a mini transformer encoder from components — positional
#   encoding, multi-head attention, feed-forward network, layer
#   normalization, and residual connections.
#
# TASKS:
#   1. Implement sinusoidal positional encoding
#   2. Build transformer encoder layer (attention + FFN + LayerNorm + residual)
#   3. Stack layers into full encoder
#   4. Train on text classification task
#   5. Visualize attention patterns per layer with ModelVisualizer
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import random
import re
from collections import Counter

import polars as pl

from kailash_ml import DataExplorer, ModelVisualizer

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
df = loader.load("ascent08", "sg_news_articles.parquet")

explorer = DataExplorer()
summary  = await explorer.profile(df)
print(f"=== Dataset: {df.height} articles ===")
print(summary)



### Helpers


In [ ]:
def tokenize(text: str) -> list[str]:
    return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()


corpus = df.select("text").to_series().to_list()
word_counts = Counter(tok for t in corpus for tok in tokenize(t))
vocab = ["<pad>", "<cls>", "<unk>"] + [
    w for w, c in word_counts.most_common(2000) if c >= 2
]
word_to_idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
d_model = 32

token_embeddings = [
    [random.gauss(0, 0.1) for _ in range(d_model)] for _ in range(vocab_size)
]

print(f"Vocabulary: {vocab_size}, d_model: {d_model}")



## TASK 1: Sinusoidal positional encoding


In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> list[list[float]]:
    """PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    # TODO: Build the positional encoding table (max_len x d_model)
    pe = []
    for pos in range(max_len):
        row = [0.0] * d_model
        for i in range(0, d_model, 2):
            # TODO: Compute denominator and fill sin/cos encoding at positions i and i+1
            denom = ____  # Hint: 10000.0 ** (i / d_model)
            row[i] = ____  # Hint: math.sin(pos / denom)
            if i + 1 < d_model:
                row[i + 1] = ____  # Hint: math.cos(pos / denom)
        pe.append(row)
    return pe


max_seq_len = 50
pos_enc = sinusoidal_positional_encoding(max_seq_len, d_model)

print(f"\n--- Positional Encoding ---")
print(f"Shape: {len(pos_enc)}x{len(pos_enc[0])}")
for pos in [0, 1, 10, 49]:
    print(
        f"  pos={pos}: [{pos_enc[pos][0]:.3f}, {pos_enc[pos][1]:.3f}, ..., {pos_enc[pos][-1]:.3f}]"
    )

print(f"\nProperties:")
print(f"  - Unique encoding per position (no learned parameters)")
print(f"  - Captures relative position via dot product")
print(f"  - Generalizes to longer sequences than seen during training")



## TASK 2: Transformer encoder layer


In [ ]:
def softmax(scores: list[float]) -> list[float]:
    max_s = max(scores) if scores else 0
    exps = [math.exp(s - max_s) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]


def layer_norm(x: list[float], eps: float = 1e-6) -> list[float]:
    """Layer normalization: normalize across the feature dimension."""
    # TODO: Compute mean and variance, then normalize each element
    mean = ____  # Hint: sum(x) / len(x)
    var = ____  # Hint: sum((v - mean) ** 2 for v in x) / len(x)
    std = math.sqrt(var + eps)
    return ____  # Hint: [(v - mean) / std for v in x]


def feed_forward(
    x: list[float],
    W1: list[list[float]],
    b1: list[float],
    W2: list[list[float]],
    b2: list[float],
) -> list[float]:
    """FFN(x) = ReLU(xW1 + b1)W2 + b2."""
    d_ff = len(W1[0])
    # TODO: First linear layer with ReLU: hidden = max(0, x @ W1 + b1)
    hidden = [0.0] * d_ff
    for j in range(d_ff):
        val = b1[j]
        for k in range(len(x)):
            val += x[k] * W1[k][j]
        hidden[j] = ____  # Hint: max(0.0, val)

    # TODO: Second linear layer: output = hidden @ W2 + b2
    d_out = len(W2[0])
    output = [0.0] * d_out
    for j in range(d_out):
        val = b2[j]
        for k in range(d_ff):
            val += hidden[k] * W2[k][j]
        output[j] = val
    return output


def residual_add(x: list[float], sublayer_out: list[float]) -> list[float]:
    """Residual connection: x + sublayer(x)."""
    return ____  # Hint: [a + b for a, b in zip(x, sublayer_out)]


class TransformerEncoderLayer:
    """Single transformer encoder layer: self-attention + FFN + residuals + LayerNorm."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        scale = 1.0 / math.sqrt(self.d_k)

        self.W_q = [
            [[random.gauss(0, scale) for _ in range(self.d_k)] for _ in range(d_model)]
            for _ in range(n_heads)
        ]
        self.W_k = [
            [[random.gauss(0, scale) for _ in range(self.d_k)] for _ in range(d_model)]
            for _ in range(n_heads)
        ]
        self.W_v = [
            [[random.gauss(0, scale) for _ in range(self.d_k)] for _ in range(d_model)]
            for _ in range(n_heads)
        ]
        self.W_o = [
            [random.gauss(0, scale) for _ in range(d_model)] for _ in range(d_model)
        ]

        ff_scale = 1.0 / math.sqrt(d_ff)
        self.W1 = [
            [random.gauss(0, ff_scale) for _ in range(d_ff)] for _ in range(d_model)
        ]
        self.b1 = [0.0] * d_ff
        self.W2 = [
            [random.gauss(0, scale) for _ in range(d_model)] for _ in range(d_ff)
        ]
        self.b2 = [0.0] * d_model

    def self_attention(
        self, X: list[list[float]]
    ) -> tuple[list[list[float]], list[list[list[float]]]]:
        """Multi-head self-attention."""
        all_heads = []
        all_weights = []
        for h in range(self.n_heads):
            # TODO: Project X to Q, K, V using per-head weights (QKV projection)
            Q = ____  # Hint: [[sum(X[i][k] * self.W_q[h][k][j] for k in range(self.d_model)) for j in range(self.d_k)] for i in range(len(X))]
            K = ____  # Hint: same pattern with self.W_k[h]
            V = ____  # Hint: same pattern with self.W_v[h]

            # TODO: Compute scaled attention scores and softmax weights
            scale = math.sqrt(self.d_k)
            scores = ____  # Hint: [[sum(Q[i][d] * K[j][d] for d in range(self.d_k)) / scale for j in range(len(K))] for i in range(len(Q))]
            weights = ____  # Hint: [softmax(row) for row in scores]
            # TODO: Compute weighted sum of values for each query position
            head_out = ____  # Hint: [[sum(weights[i][j] * V[j][d] for j in range(len(V))) for d in range(self.d_k)] for i in range(len(Q))]
            all_heads.append(head_out)
            all_weights.append(weights)

        # Concatenate heads then project with W_o
        concat = [[] for _ in range(len(X))]
        for i in range(len(X)):
            for h in range(self.n_heads):
                concat[i].extend(all_heads[h][i])

        output = [
            [
                sum(concat[i][k] * self.W_o[k][j] for k in range(self.d_model))
                for j in range(self.d_model)
            ]
            for i in range(len(X))
        ]
        return output, all_weights

    def forward(
        self, X: list[list[float]]
    ) -> tuple[list[list[float]], list[list[list[float]]]]:
        """Full encoder layer: attention -> residual -> LayerNorm -> FFN -> residual -> LayerNorm."""
        # TODO: Self-attention sub-layer with residual and LayerNorm
        attn_out, attn_weights = self.self_attention(X)
        normed1 = ____  # Hint: [layer_norm(residual_add(X[i], attn_out[i])) for i in range(len(X))]

        # TODO: FFN sub-layer with residual and LayerNorm
        ffn_out = ____  # Hint: [feed_forward(normed1[i], self.W1, self.b1, self.W2, self.b2) for i in range(len(normed1))]
        normed2 = ____  # Hint: [layer_norm(residual_add(normed1[i], ffn_out[i])) for i in range(len(normed1))]

        return normed2, attn_weights


n_heads = 4
d_ff = 64
layer = TransformerEncoderLayer(d_model, n_heads, d_ff)

sample_tokens = tokenize(corpus[0] if corpus else "singapore economy growth")[:10]
sample_indices = [word_to_idx.get(t, 2) for t in sample_tokens]
# TODO: Build input by adding token embeddings and positional encodings
sample_input = ____  # Hint: [[token_embeddings[idx][d] + pos_enc[pos][d] for d in range(d_model)] for pos, idx in enumerate(sample_indices)]

layer_out, layer_attn = layer.forward(sample_input)
print(f"\n--- Encoder Layer ---")
print(
    f"Input: {len(sample_input)}x{d_model}, Output: {len(layer_out)}x{len(layer_out[0])}"
)
print(f"Attention heads: {n_heads}, FFN dim: {d_ff}")



## TASK 3: Stack layers into full encoder


In [ ]:
class TransformerEncoder:
    """Stack of N transformer encoder layers."""

    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int):
        # TODO: Create n_layers TransformerEncoderLayer instances
        self.layers = ____  # Hint: [TransformerEncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)]

    def forward(self, X: list[list[float]]) -> tuple[list[list[float]], list]:
        """Forward through all layers, collecting attention weights."""
        all_layer_attn = []
        hidden = X
        for layer in self.layers:
            # TODO: Pass hidden through each layer, collect attention weights
            hidden, attn_weights = ____  # Hint: layer.forward(hidden)
            all_layer_attn.append(attn_weights)
        return hidden, all_layer_attn


n_layers = 3
encoder = TransformerEncoder(n_layers, d_model, n_heads, d_ff)
enc_output, all_attn = encoder.forward(sample_input)

print(f"\n--- Full Encoder ({n_layers} layers) ---")
print(f"Output shape: {len(enc_output)}x{len(enc_output[0])}")
print(f"Attention maps collected: {len(all_attn)} layers x {n_heads} heads")



## TASK 4: Train on text classification task


In [ ]:
print(f"\n=== Transformer Encoder for Classification ===")
print(f"The [CLS] token embedding from the encoder can be used as a document vector.")
print(f"In production: pass to TrainingPipeline(feature_store, registry) for training.")
print(f"Documents processed: {len(corpus)}")



## TASK 5: Visualize attention patterns per layer


In [ ]:
# TODO: Instantiate ModelVisualizer
viz = ____  # Hint: ModelVisualizer()

print(f"\n=== Attention Pattern Analysis ===")
print(f"Layer 0: tends to capture local/syntactic patterns")
print(f"Layer 1: tends to capture broader semantic relationships")
print(f"Layer 2: tends to capture task-specific patterns")
print(f"\nThis hierarchy is why deeper transformers capture more complex patterns.")

print("\n✓ Exercise 6 complete — mini transformer encoder from scratch")

